# 01. 원본 데이터 현황 점검

이 Notebook은 전처리 규칙을 결정하기 전에 원본 데이터의 구조와 품질을 확인한다.

**점검 범위**

- 2024~2026 월별 라벨 CSV의 파일 구성, 스키마, 날짜, 물량, 이벤트 라벨
- 우체국·행정동 참조 CSV 4개의 구조, 키 중복, 결측치
- 서로 중복되는 참조 파일의 공통 컬럼 값 일치 여부

**중요:** 이 Notebook은 원본 파일을 읽기만 하며 수정·이동·삭제하거나 결과 파일을 저장하지 않는다.

In [1]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)


def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "data" / "raw").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. postcast 또는 notebooks 폴더에서 실행하세요."
    )


PROJECT_ROOT = find_project_root()
MONTHLY_DIR = PROJECT_ROOT / "data" / "raw" / "monthly_labels"
REFERENCE_DIR = PROJECT_ROOT / "data" / "raw" / "reference"

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"월별 라벨 경로: {MONTHLY_DIR}")
print(f"참조 데이터 경로: {REFERENCE_DIR}")

프로젝트 루트: /Users/hyewon/Documents/postcast
월별 라벨 경로: /Users/hyewon/Documents/postcast/data/raw/monthly_labels
참조 데이터 경로: /Users/hyewon/Documents/postcast/data/raw/reference


In [2]:
ENCODINGS = ("utf-8-sig", "utf-8", "cp949", "euc-kr")


def nfc(value) -> str:
    return unicodedata.normalize("NFC", str(value)).strip()


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = [nfc(column) for column in result.columns]
    return result


def read_csv_auto(path: Path):
    errors = []
    for encoding in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=encoding), encoding
        except UnicodeError as error:
            errors.append(f"{encoding}: {error}")
    raise UnicodeError(f"{path.name} 인코딩 판별 실패: {errors}")


def extract_yyyymm(filename: str):
    match = re.search(r"(20\d{4})", nfc(filename))
    return match.group(1) if match else None


def fully_blank_mask(df: pd.DataFrame) -> pd.Series:
    return df.apply(
        lambda row: all(
            pd.isna(value)
            or (isinstance(value, str) and not value.strip())
            for value in row
        ),
        axis=1,
    )


monthly_files = sorted(MONTHLY_DIR.rglob("*.csv"), key=lambda path: nfc(str(path)))
reference_files = sorted(REFERENCE_DIR.glob("*.csv"), key=lambda path: nfc(path.name))

inventory_rows = []
for kind, files in (("monthly_label", monthly_files), ("reference", reference_files)):
    for path in files:
        inventory_rows.append(
            {
                "구분": kind,
                "상대경로": nfc(path.relative_to(PROJECT_ROOT)),
                "파일명": nfc(path.name),
                "폴더연도": path.parent.name if kind == "monthly_label" else None,
                "파일연월": extract_yyyymm(path.name),
                "크기_KB": round(path.stat().st_size / 1024, 1),
            }
        )

file_inventory = pd.DataFrame(inventory_rows)
display(file_inventory)
print(f"월별 라벨 CSV: {len(monthly_files)}개")
print(f"참조 CSV: {len(reference_files)}개")

,구분,상대경로,파일명,폴더연도,파일연월,크기_KB
0,monthly_label,data/raw/monthly_labels/2024/대전광역시_일반통상정보_202401_라벨링.csv,대전광역시_일반통상정보_202401_라벨링.csv,2024,202401,1.3
1,monthly_label,data/raw/monthly_labels/2024/대전광역시_일반통상정보_202402_라벨링.csv,대전광역시_일반통상정보_202402_라벨링.csv,2024,202402,1.2
2,monthly_label,data/raw/monthly_labels/2024/대전광역시_일반통상정보_202403_라벨링.csv,대전광역시_일반통상정보_202403_라벨링.csv,2024,202403,1.3
3,monthly_label,data/raw/monthly_labels/2024/대전광역시_일반통상정보_202404_라벨링.csv,대전광역시_일반통상정보_202404_라벨링.csv,2024,202404,1.3
4,monthly_label,data/raw/monthly_labels/2024/대전광역시_일반통상정보_202405_라벨링.csv,대전광역시_일반통상정보_202405_라벨링.csv,2024,202405,1.3
5,monthly_label,data/raw/monthly_labels/2024/대전광역시_일반통상정보_202406_라벨링.csv,대전광역시_일반통상정보_202406_라벨링.csv,2024,202406,1.1
6,monthly_label,data/raw/monthly_labels/2024/대전광역시_일반통상정보_202407_라벨링.csv,대전광역시_일반통상정보_202407_라벨링.csv,2024,202407,1.4
7,monthly_label,data/raw/monthly_labels/2024/대전광역시_일반통상정보_202408_라벨링.csv,대전광역시_일반통상정보_202408_라벨링.csv,2024,202408,1.2
8,monthly_label,data/raw/monthly_labels/2024/대전광역시_일반통상정보_202409_라벨링.csv,대전광역시_일반통상정보_202409_라벨링.csv,2024,202409,1.1
9,monthly_label,data/raw/monthly_labels/2024/대전광역시_일반통상정보_202410_라벨링.csv,대전광역시_일반통상정보_202410_라벨링.csv,2024,202410,1.2


월별 라벨 CSV: 30개
참조 CSV: 4개


## 1. 월별 파일 구성 점검

파일명에서 추출한 연월을 기준으로 2024년 1월부터 2026년 6월까지의 월이 모두 존재하는지 확인한다.
파일명은 `_YYYYMM_라벨링.csv` 형식을 권장하되, 이 단계에서는 이름을 변경하지 않는다.

In [3]:
expected_months = set(
    pd.period_range("2024-01", "2026-06", freq="M").strftime("%Y%m")
)
found_months = {
    extract_yyyymm(path.name)
    for path in monthly_files
    if extract_yyyymm(path.name)
}

missing_months = sorted(expected_months - found_months)
unexpected_months = sorted(found_months - expected_months)
duplicate_months = (
    pd.Series([extract_yyyymm(path.name) for path in monthly_files])
    .value_counts()
    .loc[lambda series: series > 1]
    .to_dict()
)
unusual_filenames = [
    nfc(path.name)
    for path in monthly_files
    if not re.search(r"_20\d{4}_라벨링\.csv$", nfc(path.name))
]
folder_mismatches = [
    nfc(path.relative_to(PROJECT_ROOT))
    for path in monthly_files
    if extract_yyyymm(path.name)
    and extract_yyyymm(path.name)[:4] != path.parent.name
]

month_check = pd.DataFrame(
    [
        {"점검항목": "기대 월 개수", "결과": len(expected_months)},
        {"점검항목": "발견 월 개수", "결과": len(found_months)},
        {"점검항목": "누락 월", "결과": missing_months or "없음"},
        {"점검항목": "범위 밖 월", "결과": unexpected_months or "없음"},
        {"점검항목": "중복 월", "결과": duplicate_months or "없음"},
        {"점검항목": "파일명 형식 이상", "결과": unusual_filenames or "없음"},
        {"점검항목": "연도 폴더 불일치", "결과": folder_mismatches or "없음"},
    ]
)
display(month_check)

,점검항목,결과
0,기대 월 개수,30
1,발견 월 개수,30
2,누락 월,없음
3,범위 밖 월,없음
4,중복 월,없음
5,파일명 형식 이상,없음
6,연도 폴더 불일치,없음


## 2. 월별 라벨 CSV 품질 점검

파일별로 다음 항목을 검사한다.

- 행·열 개수와 인코딩
- 완전히 비어 있는 행
- `Unnamed` 컬럼
- 날짜 파싱 실패, 파일 연월과 날짜 불일치, 파일 내부 중복 날짜
- `접수통수`의 숫자 변환 실패 및 음수
- 이벤트 컬럼의 `0/1` 이외 값
- 스키마 차이

In [4]:
DATE_CANDIDATES = ("일자", "날짜", "접수일자")
VOLUME_CANDIDATES = ("접수통수", "접수물량", "물량")
NON_EVENT_COLUMNS = {
    "지역", "접수지역", "시도", "시구", "일자", "날짜", "접수일자",
    "요일", "접수통수", "접수물량", "물량",
}


def first_existing(columns, candidates):
    return next((candidate for candidate in candidates if candidate in columns), None)


monthly_audit_rows = []
monthly_date_rows = []

for path in monthly_files:
    raw_df, encoding = read_csv_auto(path)
    raw_df = normalize_columns(raw_df)
    blank_mask = fully_blank_mask(raw_df)
    df = raw_df.loc[~blank_mask].copy()

    columns = list(df.columns)
    unnamed_columns = [column for column in columns if column.startswith("Unnamed")]
    date_column = first_existing(columns, DATE_CANDIDATES)
    volume_column = first_existing(columns, VOLUME_CANDIDATES)
    file_yyyymm = extract_yyyymm(path.name)

    invalid_dates = 0
    duplicate_dates = 0
    date_month_mismatches = 0
    min_date = None
    max_date = None

    if date_column:
        parsed_dates = pd.to_datetime(df[date_column], errors="coerce")
        invalid_dates = int(parsed_dates.isna().sum())
        valid_dates = parsed_dates.dropna()
        duplicate_dates = int(valid_dates.duplicated().sum())
        if not valid_dates.empty:
            min_date = valid_dates.min().date().isoformat()
            max_date = valid_dates.max().date().isoformat()
            if file_yyyymm:
                date_month_mismatches = int(
                    (valid_dates.dt.strftime("%Y%m") != file_yyyymm).sum()
                )
            monthly_date_rows.extend(
                {
                    "날짜": date,
                    "파일명": nfc(path.name),
                }
                for date in valid_dates
            )

    volume_non_numeric = None
    volume_negative = None
    if volume_column:
        numeric_volume = pd.to_numeric(df[volume_column], errors="coerce")
        volume_non_numeric = int(
            (numeric_volume.isna() & df[volume_column].notna()).sum()
        )
        volume_negative = int((numeric_volume < 0).sum())

    event_columns = [
        column
        for column in columns
        if column not in NON_EVENT_COLUMNS and not column.startswith("Unnamed")
    ]
    invalid_event_columns = []
    for column in event_columns:
        numeric_event = pd.to_numeric(df[column], errors="coerce")
        non_blank = df[column].notna()
        invalid_mask = non_blank & (
            numeric_event.isna() | ~numeric_event.isin([0, 1])
        )
        if invalid_mask.any():
            invalid_event_columns.append(column)

    issues = []
    if int(blank_mask.sum()):
        issues.append("완전 공백 행")
    if unnamed_columns:
        issues.append("Unnamed 컬럼")
    if date_column is None:
        issues.append("날짜 컬럼 없음")
    if invalid_dates:
        issues.append("날짜 파싱 실패")
    if duplicate_dates:
        issues.append("파일 내부 중복 날짜")
    if date_month_mismatches:
        issues.append("파일 연월 불일치")
    if volume_column is None:
        issues.append("접수통수 컬럼 없음")
    if volume_non_numeric:
        issues.append("접수통수 숫자 변환 실패")
    if volume_negative:
        issues.append("접수통수 음수")
    if invalid_event_columns:
        issues.append("이벤트 0/1 위반")

    monthly_audit_rows.append(
        {
            "파일연월": file_yyyymm,
            "파일명": nfc(path.name),
            "인코딩": encoding,
            "원본행수": len(raw_df),
            "유효행수": len(df),
            "완전공백행": int(blank_mask.sum()),
            "컬럼수": len(columns),
            "Unnamed컬럼": ", ".join(unnamed_columns) or None,
            "최소날짜": min_date,
            "최대날짜": max_date,
            "날짜파싱실패": invalid_dates,
            "파일내중복날짜": duplicate_dates,
            "파일연월불일치": date_month_mismatches,
            "접수통수숫자실패": volume_non_numeric,
            "접수통수음수": volume_negative,
            "이벤트0_1위반컬럼": ", ".join(invalid_event_columns) or None,
            "스키마": " | ".join(columns),
            "판정": "확인 필요: " + ", ".join(issues) if issues else "정상",
        }
    )

monthly_audit = (
    pd.DataFrame(monthly_audit_rows)
    .sort_values(["파일연월", "파일명"])
    .reset_index(drop=True)
)
display(monthly_audit.drop(columns=["스키마"]))

,파일연월,파일명,인코딩,원본행수,유효행수,완전공백행,컬럼수,Unnamed컬럼,최소날짜,최대날짜,날짜파싱실패,파일내중복날짜,파일연월불일치,접수통수숫자실패,접수통수음수,이벤트0_1위반컬럼,판정
0,202401,대전광역시_일반통상정보_202401_라벨링.csv,utf-8-sig,22,22,0,11,NaN,2024-01-02,2024-01-31,0,0,0,0,0,None,정상
1,202402,대전광역시_일반통상정보_202402_라벨링.csv,utf-8-sig,25,19,6,11,NaN,2024-02-01,2024-02-29,0,0,0,0,0,None,확인 필요: 완전 공백 행
2,202403,대전광역시_일반통상정보_202403_라벨링.csv,utf-8-sig,22,22,0,11,NaN,2024-03-04,2024-03-31,0,0,0,0,0,None,정상
3,202404,대전광역시_일반통상정보_202404_라벨링.csv,utf-8-sig,25,21,4,12,Unnamed: 11,2024-04-01,2024-04-30,0,0,0,0,0,None,"확인 필요: 완전 공백 행, Unnamed 컬럼"
4,202405,대전광역시_일반통상정보_202405_라벨링.csv,utf-8-sig,22,21,1,11,NaN,2024-05-01,2024-05-31,0,0,0,0,0,None,확인 필요: 완전 공백 행
5,202406,대전광역시_일반통상정보_202406_라벨링.csv,utf-8-sig,19,19,0,11,NaN,2024-06-03,2024-06-28,0,0,0,0,0,None,정상
6,202407,대전광역시_일반통상정보_202407_라벨링.csv,utf-8-sig,24,23,1,11,NaN,2024-07-01,2024-07-31,0,0,0,0,0,None,확인 필요: 완전 공백 행
7,202408,대전광역시_일반통상정보_202408_라벨링.csv,utf-8-sig,21,21,0,11,NaN,2024-08-01,2024-08-30,0,0,0,0,0,None,정상
8,202409,대전광역시_일반통상정보_202409_라벨링.csv,utf-8-sig,18,18,0,11,NaN,2024-09-02,2024-09-30,0,0,0,0,0,None,정상
9,202410,대전광역시_일반통상정보_202410_라벨링.csv,utf-8-sig,20,20,0,11,NaN,2024-10-02,2024-10-31,0,0,0,0,0,None,정상


In [5]:
schema_summary = (
    monthly_audit.groupby("스키마", dropna=False)
    .agg(
        파일수=("파일명", "size"),
        대상파일=("파일명", lambda values: "\n".join(values)),
    )
    .reset_index()
)
schema_summary.insert(0, "스키마그룹", range(1, len(schema_summary) + 1))
display(schema_summary)

all_monthly_dates = pd.DataFrame(monthly_date_rows)
if not all_monthly_dates.empty:
    cross_file_duplicates = (
        all_monthly_dates.groupby("날짜")
        .agg(
            등장횟수=("파일명", "size"),
            대상파일=("파일명", lambda values: ", ".join(sorted(set(values)))),
        )
        .query("등장횟수 > 1")
        .reset_index()
    )
else:
    cross_file_duplicates = pd.DataFrame(
        columns=["날짜", "등장횟수", "대상파일"]
    )

print(f"스키마 그룹 수: {len(schema_summary)}")
print(f"파일 간 중복 날짜 수: {len(cross_file_duplicates)}")
display(cross_file_duplicates.head(30))

,스키마그룹,스키마,파일수,대상파일
0,1,접수일자 | 접수지역 | 접수통수 | 요일 | 제1기분 자동차세 | 재산세 (건축) | 정기분 주민세 | 주민세 (사업소분) | 재산세 (토지) | 제2기분 자동차세 | 사회보험료 통합,7,대전광역시_일반통상정보_202409_라벨링.csv\n대전광역시_일반통상정보_202410_라벨링.csv\n대전광역시_일반통상정보_202411_라벨링.csv\n대전광역시_일반통상정보_202412_라벨링.csv\n...
1,2,접수일자 | 접수지역 | 접수통수 | 요일 | 제1기분 자동차세 | 재산세 (건축) | 정기분 주민세 | 주민세(사업소분) | 재산세(토지) | 제2기분 자동차세 | 사회보험료 통합,1,대전광역시_일반통상정보_202504_라벨링.csv
2,3,접수일자 | 접수지역 | 접수통수 | 요일 | 제1기분 자동차세 | 재산세(건축) | 정기분 주민세 | 주민세(사업소분) | 재산세(토지) | 제2기분 자동차세 | 사회보험료 통합,16,대전광역시_일반통상정보_202405_라벨링.csv\n대전광역시_일반통상정보_202408_라벨링.csv\n대전광역시_일반통상정보_202505_라벨링.csv\n대전광역시_일반통상정보_202506_라벨링.csv\n...
3,4,접수일자 | 접수지역 | 접수통수 | 요일 | 제1기분 자동차세2 | 재산세(건축) | 정기분 주민세 | 주민세(사업소분) | 재산세(토지) | 제2기분 자동차세 | 사회보험료 통합,5,대전광역시_일반통상정보_202401_라벨링.csv\n대전광역시_일반통상정보_202402_라벨링.csv\n대전광역시_일반통상정보_202403_라벨링.csv\n대전광역시_일반통상정보_202406_라벨링.csv\n...
4,5,접수일자 | 접수지역 | 접수통수 | 요일 | 제1기분 자동차세2 | 재산세(건축) | 정기분 주민세 | 주민세(사업소분) | 재산세(토지) | 제2기분 자동차세 | 사회보험료 통합 | Unnamed: 11,1,대전광역시_일반통상정보_202404_라벨링.csv


스키마 그룹 수: 5
파일 간 중복 날짜 수: 0


,날짜,등장횟수,대상파일


## 3. 우체국·행정동 참조 CSV 점검

참조 파일별 행·열 수, 결측 셀, 완전 중복 행, 후보 키 중복과 `Unnamed` 컬럼을 확인한다.

In [6]:
reference_frames = {}
reference_audit_rows = []

for path in reference_files:
    raw_df, encoding = read_csv_auto(path)
    df = normalize_columns(raw_df)
    filename = nfc(path.name)
    reference_frames[filename] = df

    if "통상배달국명" in df.columns:
        key_columns = ["통상배달국명"]
    elif {"시구", "행정동명"}.issubset(df.columns):
        key_columns = ["시구", "행정동명"]
    elif "행정동" in df.columns:
        key_columns = ["행정동"]
    else:
        key_columns = []

    duplicate_keys = (
        int(df.duplicated(subset=key_columns).sum()) if key_columns else None
    )
    unnamed_columns = [
        column for column in df.columns if column.startswith("Unnamed")
    ]

    reference_audit_rows.append(
        {
            "파일명": filename,
            "인코딩": encoding,
            "행수": len(df),
            "컬럼수": len(df.columns),
            "결측셀": int(df.isna().sum().sum()),
            "완전중복행": int(df.duplicated().sum()),
            "후보키": " + ".join(key_columns) if key_columns else None,
            "후보키중복": duplicate_keys,
            "Unnamed컬럼": ", ".join(unnamed_columns) or None,
            "컬럼목록": " | ".join(df.columns),
        }
    )

reference_audit = pd.DataFrame(reference_audit_rows)
display(reference_audit)

,파일명,인코딩,행수,컬럼수,결측셀,완전중복행,후보키,후보키중복,Unnamed컬럼,컬럼목록
0,대전_통상배달국_좌표.csv,utf-8-sig,5,7,0,0,통상배달국명,0,NaN,통상배달국명 | 우체국명 | 공식주소 | 반환주소 | 위도 | 경도 | 검증결과
1,우체국별_정보3.csv,utf-8-sig,5,10,0,0,통상배달국명,0,Unnamed: 0,Unnamed: 0 | 통상배달국명 | 우체국명 | 공식주소 | 반환주소 | 위도 | 경도 | 검증결과 | 추정집배원수 | 관할행정동
2,행정동별_사업체수_면적_세대_수(csv).csv,utf-8-sig,82,5,0,0,행정동,0,NaN,행정동 | 사업체수 | 시구 | 세대수 | 면적 (㎢)
3,행정동별_정보3.csv,utf-8-sig,82,21,0,0,시구 + 행정동명,0,Unnamed: 0,Unnamed: 0 | 행정동명 | 사업체수 | 시구 | 세대수 | 면적 (㎢) | 주소수 | 담당 통상배달국명 | 담당 집배원수(추정) | 주소밀도 | 행정동_위도 | 행정동_경도 | 담당 우체국명 | 담당...


## 4. 중복 참조 파일의 공통 값 비교

- `대전_통상배달국_좌표.csv`와 `우체국별_정보3.csv`
- `행정동별_사업체수_면적_세대_수(csv).csv`와 `행정동별_정보3.csv`

공통 키로 연결한 뒤 공통 컬럼의 값이 일치하는지 확인한다.

In [7]:
def normalized_for_compare(df: pd.DataFrame) -> pd.DataFrame:
    result = normalize_columns(df)
    result = result.drop(
        columns=[
            column
            for column in result.columns
            if column.startswith("Unnamed")
        ],
        errors="ignore",
    )
    for column in result.columns:
        if pd.api.types.is_string_dtype(result[column]):
            result[column] = result[column].map(
                lambda value: nfc(value) if pd.notna(value) else value
            )
    return result


def compare_shared_columns(left, right, keys, comparison_name):
    left = normalized_for_compare(left)
    right = normalized_for_compare(right)
    common_columns = [
        column
        for column in left.columns
        if column in right.columns and column not in keys
    ]
    merged = left.merge(
        right,
        on=keys,
        how="outer",
        suffixes=("_왼쪽", "_오른쪽"),
        indicator=True,
    )

    result_rows = [
        {
            "비교": comparison_name,
            "항목": "키 포함 범위",
            "결과": merged["_merge"].value_counts().to_dict(),
        }
    ]

    for column in common_columns:
        left_values = merged[f"{column}_왼쪽"]
        right_values = merged[f"{column}_오른쪽"]
        left_numeric = pd.to_numeric(left_values, errors="coerce")
        right_numeric = pd.to_numeric(right_values, errors="coerce")

        both_blank = left_values.isna() & right_values.isna()
        numeric_rows = left_numeric.notna() & right_numeric.notna()
        equal = pd.Series(False, index=merged.index)
        equal.loc[both_blank] = True
        equal.loc[numeric_rows] = np.isclose(
            left_numeric.loc[numeric_rows],
            right_numeric.loc[numeric_rows],
            rtol=1e-9,
            atol=1e-9,
        )

        text_rows = ~both_blank & ~numeric_rows
        equal.loc[text_rows] = (
            left_values.loc[text_rows].fillna("").astype(str)
            == right_values.loc[text_rows].fillna("").astype(str)
        )

        result_rows.append(
            {
                "비교": comparison_name,
                "항목": column,
                "결과": int((~equal).sum()),
            }
        )

    return pd.DataFrame(result_rows)


def find_reference(partial_name):
    return next(
        (
            frame
            for filename, frame in reference_frames.items()
            if partial_name in filename
        ),
        None,
    )


office_coordinates = find_reference("대전_통상배달국_좌표")
office_information = find_reference("우체국별_정보3")
admin_statistics = find_reference("행정동별_사업체수_면적_세대_수")
admin_information = find_reference("행정동별_정보3")

comparison_results = []

if office_coordinates is not None and office_information is not None:
    comparison_results.append(
        compare_shared_columns(
            office_coordinates,
            office_information,
            ["통상배달국명"],
            "배달국 좌표 ↔ 우체국 정보",
        )
    )

if admin_statistics is not None and admin_information is not None:
    admin_statistics_for_compare = admin_statistics.rename(
        columns={"행정동": "행정동명"}
    )
    comparison_results.append(
        compare_shared_columns(
            admin_statistics_for_compare,
            admin_information,
            ["시구", "행정동명"],
            "행정동 기본통계 ↔ 행정동 정보",
        )
    )

source_comparison = (
    pd.concat(comparison_results, ignore_index=True)
    if comparison_results
    else pd.DataFrame(columns=["비교", "항목", "결과"])
)
display(source_comparison)

,비교,항목,결과
0,배달국 좌표 ↔ 우체국 정보,키 포함 범위,"{'both': 5, 'left_only': 0, 'right_only': 0}"
1,배달국 좌표 ↔ 우체국 정보,우체국명,0
2,배달국 좌표 ↔ 우체국 정보,공식주소,0
3,배달국 좌표 ↔ 우체국 정보,반환주소,0
4,배달국 좌표 ↔ 우체국 정보,위도,0
5,배달국 좌표 ↔ 우체국 정보,경도,0
6,배달국 좌표 ↔ 우체국 정보,검증결과,0
7,행정동 기본통계 ↔ 행정동 정보,키 포함 범위,"{'both': 82, 'left_only': 0, 'right_only': 0}"
8,행정동 기본통계 ↔ 행정동 정보,사업체수,0
9,행정동 기본통계 ↔ 행정동 정보,세대수,0


## 5. 점검 요약

아래 표는 다음 전처리 단계에서 결정하거나 수정해야 할 항목을 요약한다.
이 표가 출력되어도 원본 파일에는 어떠한 변경도 발생하지 않는다.

In [8]:
check_needed_files = monthly_audit.loc[
    monthly_audit["판정"] != "정상", ["파일연월", "파일명", "판정"]
]

summary = pd.DataFrame(
    [
        {
            "점검항목": "월별 라벨 파일",
            "결과": f"{len(monthly_files)}개",
            "상태": "정상" if len(monthly_files) == 30 else "확인 필요",
        },
        {
            "점검항목": "누락 월",
            "결과": missing_months or "없음",
            "상태": "정상" if not missing_months else "확인 필요",
        },
        {
            "점검항목": "파일명 형식 이상",
            "결과": unusual_filenames or "없음",
            "상태": "정상" if not unusual_filenames else "확인 필요",
        },
        {
            "점검항목": "월별 스키마 그룹",
            "결과": f"{len(schema_summary)}개",
            "상태": "정상" if len(schema_summary) == 1 else "확인 필요",
        },
        {
            "점검항목": "완전 공백 행 합계",
            "결과": int(monthly_audit["완전공백행"].sum()),
            "상태": (
                "정상"
                if int(monthly_audit["완전공백행"].sum()) == 0
                else "확인 필요"
            ),
        },
        {
            "점검항목": "파일 간 중복 날짜",
            "결과": len(cross_file_duplicates),
            "상태": "정상" if cross_file_duplicates.empty else "확인 필요",
        },
        {
            "점검항목": "참조 CSV",
            "결과": f"{len(reference_files)}개",
            "상태": "정상" if len(reference_files) == 4 else "확인 필요",
        },
    ]
)

display(summary)
display(Markdown("### 세부 확인 대상 파일"))
display(check_needed_files)

print("점검 완료: 원본 파일은 수정하지 않았습니다.")

,점검항목,결과,상태
0,월별 라벨 파일,30개,정상
1,누락 월,없음,정상
2,파일명 형식 이상,없음,정상
3,월별 스키마 그룹,5개,확인 필요
4,완전 공백 행 합계,15,확인 필요
5,파일 간 중복 날짜,0,정상
6,참조 CSV,4개,정상


### 세부 확인 대상 파일

,파일연월,파일명,판정
1,202402,대전광역시_일반통상정보_202402_라벨링.csv,확인 필요: 완전 공백 행
3,202404,대전광역시_일반통상정보_202404_라벨링.csv,"확인 필요: 완전 공백 행, Unnamed 컬럼"
4,202405,대전광역시_일반통상정보_202405_라벨링.csv,확인 필요: 완전 공백 행
6,202407,대전광역시_일반통상정보_202407_라벨링.csv,확인 필요: 완전 공백 행
16,202505,대전광역시_일반통상정보_202505_라벨링.csv,확인 필요: 완전 공백 행
19,202508,대전광역시_일반통상정보_202508_라벨링.csv,확인 필요: 완전 공백 행
28,202605,대전광역시_일반통상정보_202605_라벨링.csv,확인 필요: 완전 공백 행


점검 완료: 원본 파일은 수정하지 않았습니다.
